In [28]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
import plotly.graph_objs as go

In [2]:
df_cleaned = pd.read_csv('df_cleaned.csv')

In [3]:
# Change to let the user select their stocks later
selected_stocks = ['AAPL', 'MSFT', 'GOOG']

df_stocks = df_cleaned[selected_stocks]

In [26]:
stock_options = [{"label": stock, "value": stock} for stock in sorted(df_stocks)]

app = Dash(__name__)

app.layout = html.Div([
    html.H4("Stock Price Analysis"),
    dcc.Graph(id="time-series-chart"),

    html.Div([
        html.Div([
            html.P("Select Stock:"),
            dcc.Dropdown(id="stock1", options=stock_options, value="AMZN", clearable=True),
        ], style={"margin": "0 10px", "width": "30%"}),

        html.Div([
            html.P("Select Percentage of Portfolio:"),
            dcc.Input(id="pct1", type="number", min=0, max=100, step=1, debounce=True),
        ], style={"margin": "0 10px", "width": "30%"}),

    ], style={"display": "flex", "justify-content": "space-between"}),

    html.Div([
        html.Div([
            dcc.Dropdown(id="stock2", options=stock_options, value="AMZN", clearable=True),
        ], style={"margin": "0 10px", "width": "30%"}),

        html.Div([
            dcc.Input(id="pct2", type="number", min=0, max=100, step=1, debounce=True),
        ], style={"margin": "0 10px", "width": "30%"}),

    ], style={"display": "flex", "justify-content": "space-between"}),


    html.Div([
        html.Div([
            dcc.Dropdown(id="stock3", options=stock_options, value="AMZN", clearable=True),
        ], style={"margin": "0 10px", "width": "30%"}),

        html.Div([
            dcc.Input(id="pct3", type="number", min=0, max=100, step=1, debounce=True),
        ], style={"margin": "0 10px", "width": "30%"}),

    ], style={"display": "flex", "justify-content": "space-between"})
])

@app.callback(
    Output("time-series-chart", "figure"),
    Input("stock1", "value"),
    Input("stock2", "value"),
    Input("stock3", "value")
)

def display_monte_carlo(s1,s2,s3):
    
    selected = [s for s in [s1, s2, s3] if s]

    if not selected:
        return px.line(title="No stocks selected")

    df_plot = df_cleaned[["Date"] + selected].copy()

    #set parameters
    weights = np.random.random(len(df_plot.columns))
    weights = np.ones(len(df_plot.columns))
    weights /= np.sum(weights)
    
    mc_sims = 4000 # number of simulations
    T = 100 #timeframe in days
    initialPortfolio = 10000

    
    returns = df_stocks.pct_change().dropna()
    meanReturns = returns.mean()
    covMatrix = returns.cov()
    returns['portfolio'] = returns.dot(weights)

    meanM = np.full(shape=(T, len(weights)), fill_value=meanReturns)
    meanM = meanM.T
    
    portfolio_sims = np.full(shape=(T, mc_sims), fill_value=0.0)
    
    for m in range(0, mc_sims):
        # MC loops
        Z = np.random.normal(size=(T, len(weights)))
        L = np.linalg.cholesky(covMatrix)
        dailyReturns = meanM + np.inner(L, Z)
        portfolio_sims[:,m] = np.cumprod(np.inner(weights, dailyReturns.T)+1)*initialPortfolio


    plt.plot(portfolio_sims)
    plt.ylabel('Portfolio Value ($)')
    plt.xlabel('Days')
    plt.title('MC simulation of a stock portfolio')
    plt.show()

    return


if __name__ == '__main__':
    app.run(debug=True)

In [32]:
stock_options = [{"label": stock, "value": stock} for stock in sorted(df_stocks.columns)]

app = Dash(__name__)

app.layout = html.Div([
    html.H4("Stock Price Analysis"),
    dcc.Graph(id="time-series-chart"),

    # Stock 1
    html.Div([
        dcc.Dropdown(id="stock1", options=stock_options, value="AMZN", clearable=True),
        dcc.Input(id="pct1", type="number", min=0, max=100, step=1, debounce=True),
    ], style={"display": "flex"}),

    # Stock 2
    html.Div([
        dcc.Dropdown(id="stock2", options=stock_options, value="AAPL", clearable=True),
        dcc.Input(id="pct2", type="number", min=0, max=100, step=1, debounce=True),
    ], style={"display": "flex"}),

    # Stock 3
    html.Div([
        dcc.Dropdown(id="stock3", options=stock_options, value="GOOG", clearable=True),
        dcc.Input(id="pct3", type="number", min=0, max=100, step=1, debounce=True),
    ], style={"display": "flex"}),
])

@app.callback(
    Output("time-series-chart", "figure"),
    Input("stock1", "value"),
    Input("pct1", "value"),
    Input("stock2", "value"),
    Input("pct2", "value"),
    Input("stock3", "value"),
    Input("pct3", "value"),
)
def display_monte_carlo(s1, p1, s2, p2, s3, p3):
    stocks = [s1, s2, s3]
    pcts = [p1, p2, p3]

    selected = [(s, p) for s, p in zip(stocks, pcts) if s and p is not None]
    if not selected:
        return go.Figure()

    tickers = [s for s, _ in selected]
    weights = np.array([p for _, p in selected])
    weights = weights / weights.sum()

    returns = df_stocks[tickers].pct_change().dropna()
    mean_returns = returns.mean()
    cov_matrix = returns.cov()

    T = 100
    mc_sims = 500
    initial_portfolio = 10000

    portfolio_sims = np.zeros((T, mc_sims))

    for i in range(mc_sims):
        Z = np.random.normal(size=(T, len(weights)))
        L = np.linalg.cholesky(cov_matrix)
        daily_returns = np.dot(Z, L.T) + mean_returns.values
        portfolio_returns = np.cumprod(np.dot(daily_returns, weights) + 1) * initial_portfolio
        portfolio_sims[:, i] = portfolio_returns

    # Plot with Plotly
    fig = go.Figure()
    for i in range(min(50, mc_sims)):  # Only show 50 lines to avoid lag
        fig.add_trace(go.Scatter(y=portfolio_sims[:, i], mode='lines', line=dict(width=1), opacity=0.3, showlegend=False))
    
    fig.update_layout(
        title="Monte Carlo Simulation of Portfolio Value",
        xaxis_title="Days",
        yaxis_title="Portfolio Value ($)",
    )
    return fig

if __name__ == '__main__':
    app.run(debug=True)


In [36]:
# Start with priors, make changes at the margin (have a maximum change, account for friction)
# Bayesian approach?